In [6]:
from pathlib import Path
from datetime import date
import sys

import polars as pl

repo_root = Path.cwd()
if not (repo_root / "qts").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from qts.data._schemas import FUTURES_INTRADAY_OHLCV_COLUMNS
from qts.data.manager import DataManager
from qts.data.sources.vnstock import VnstockFuturesDataSource
from qts.data.storage.duckdb import DuckDBStorage
from qts.utils.paths import database_path

csv_path = repo_root / "data" / "VN30F1M.csv"
table = "vn_futures_intraday_prices"
symbol = "VNF:VN30F1M"
csv_interval = "15m"
fetch_interval = "1h"
fetch_start = date(2021, 5, 29)
fetch_end = date(2026, 5, 29)

storage = DuckDBStorage(database=str(database_path()))
manager = DataManager(
    stock_source=None,
    crypto_source=None,
    vn_futures_source=VnstockFuturesDataSource.from_env(),
    storage=storage,
)


## Load data and store

In [7]:
csv_frame = pl.DataFrame()
if csv_path.exists():
    csv_frame = (
        pl.read_csv(csv_path)
        .with_columns(
            pl.col("Time").str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S").alias("bar_time"),
            pl.lit(symbol).alias("symbol"),
            pl.lit(csv_interval).alias("interval"),
        )
        .with_columns(pl.col("bar_time").dt.date().alias("date"))
        .rename(
            {
                "Open": "open",
                "High": "high",
                "Low": "low",
                "Close": "close",
                "Volume": "volume",
            }
        )
        .select(FUTURES_INTRADAY_OHLCV_COLUMNS)
    )
    manager.upsert_bars(
        table,
        csv_frame,
        sort_by=["bar_time"],
        identity=["symbol", "interval", "bar_time"],
    )

pl.DataFrame(
    {
        "csv_path": [str(csv_path)],
        "csv_present": [csv_path.exists()],
        "csv_rows_imported": [csv_frame.height],
    }
)


csv_path,csv_present,csv_rows_imported
str,bool,i64
"""/Users/s2997726/Desktop/code/q…",true,33418


## Fetch data and store

In [8]:
fetched = await manager.get_vn_futures_ohlcv(
    [symbol],
    start=fetch_start,
    end=fetch_end,
    interval='1h',
)
if fetched.columns != FUTURES_INTRADAY_OHLCV_COLUMNS:
    raise ValueError(f"Unexpected futures intraday schema: {fetched.columns}")
print(fetched.head(10))
stored = manager.upsert_bars(
    table,
    fetched,
    sort_by=["bar_time"],
    identity=["symbol", "interval", "bar_time"],
)

fetched.select(
    pl.len().alias("fetched_rows"),
    pl.min("bar_time").alias("first_fetched_bar_time"),
    pl.max("bar_time").alias("last_fetched_bar_time"),
)


shape: (10, 9)
┌───────────────┬────────────┬─────────────┬──────────┬───┬────────┬────────┬────────┬────────┐
│ bar_time      ┆ date       ┆ symbol      ┆ interval ┆ … ┆ high   ┆ low    ┆ close  ┆ volume │
│ ---           ┆ ---        ┆ ---         ┆ ---      ┆   ┆ ---    ┆ ---    ┆ ---    ┆ ---    │
│ datetime[μs]  ┆ date       ┆ str         ┆ str      ┆   ┆ f64    ┆ f64    ┆ f64    ┆ f64    │
╞═══════════════╪════════════╪═════════════╪══════════╪═══╪════════╪════════╪════════╪════════╡
│ 2025-10-17    ┆ 2025-10-17 ┆ VNF:VN30F1M ┆ 1h       ┆ … ┆ 1988.0 ┆ 1981.0 ┆ 1986.8 ┆ 10.0   │
│ 09:00:00      ┆            ┆             ┆          ┆   ┆        ┆        ┆        ┆        │
│ 2025-10-17    ┆ 2025-10-17 ┆ VNF:VN30F1M ┆ 1h       ┆ … ┆ 1999.4 ┆ 1986.7 ┆ 1989.5 ┆ 5.0    │
│ 10:00:00      ┆            ┆             ┆          ┆   ┆        ┆        ┆        ┆        │
│ 2025-10-17    ┆ 2025-10-17 ┆ VNF:VN30F1M ┆ 1h       ┆ … ┆ 1986.8 ┆ 1986.7 ┆ 1986.7 ┆ 2.0    │
│ 11:00:00      ┆        

fetched_rows,first_fetched_bar_time,last_fetched_bar_time
u32,datetime[μs],datetime[μs]
775,2025-10-17 09:00:00,2026-05-29 14:00:00


In [9]:
where_clause = f"symbol = '{symbol}' and interval = '{fetch_interval}'"
rows = storage.query(f"select * from {table} where {where_clause}")
duplicate_key_count = storage.query(
    f"""
    select symbol, interval, bar_time, count(*) as row_count
    from {table}
    where {where_clause}
    group by 1, 2, 3
    having count(*) > 1
    """
).height
symbols = storage.query(f"select distinct symbol from {table} where {where_clause} order by symbol")["symbol"].to_list()
intervals = storage.query(f"select distinct interval from {table} where {where_clause} order by interval")["interval"].to_list()

pl.DataFrame(
    {
        "row_count": [rows.height],
        "is_sorted_by_bar_time": [rows["bar_time"].is_sorted()],
        "symbols": [", ".join(symbols)],
        "intervals": [", ".join(intervals)],
        "duplicate_key_count": [duplicate_key_count],
    }
)


row_count,is_sorted_by_bar_time,symbols,intervals,duplicate_key_count
i64,bool,str,str,i64
775,true,"""VNF:VN30F1M""","""1h""",0
